# 📊 Análisis Visual de Campañas en Redes Sociales

**Objetivo:** Explorar y visualizar el rendimiento de campañas de marketing digital en múltiples plataformas (Instagram, Facebook, TikTok, LinkedIn) para identificar patrones, canales más rentables y oportunidades de optimización.

**Dataset:** 1,200 registros sintéticos de campañas durante 12 meses  
**Herramientas:** Python · Pandas · Matplotlib · Seaborn

---

## 1. Importación de Librerías y Configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Estilo visual
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
})

PALETTE = {'Instagram': '#E1306C', 'Facebook': '#1877F2', 'TikTok': '#010101', 'LinkedIn': '#0A66C2'}
print('✅ Librerías cargadas correctamente')

## 2. Generación del Dataset Sintético

In [ ]:
np.random.seed(42)
n = 1200

plataformas = ['Instagram', 'Facebook', 'TikTok', 'LinkedIn']
tipos_campana = ['Awareness', 'Conversión', 'Retargeting', 'Engagement']
segmentos = ['18-24', '25-34', '35-44', '45-54']

# Parámetros base por plataforma (CTR, CPC, conversión)
params = {
    'Instagram':  dict(ctr_mu=3.2,  ctr_sd=0.9, cpc_mu=0.85, conv_mu=4.1),
    'Facebook':   dict(ctr_mu=2.5,  ctr_sd=0.8, cpc_mu=0.72, conv_mu=3.5),
    'TikTok':     dict(ctr_mu=4.8,  ctr_sd=1.2, cpc_mu=0.55, conv_mu=2.8),
    'LinkedIn':   dict(ctr_mu=1.8,  ctr_sd=0.5, cpc_mu=3.10, conv_mu=6.2),
}

fechas_inicio = pd.date_range('2024-01-01', periods=n, freq='7H')
plataforma_col = np.random.choice(plataformas, n, p=[0.35, 0.30, 0.25, 0.10])

registros = []
for i in range(n):
    p = plataforma_col[i]
    pr = params[p]
    presupuesto = np.random.choice([500, 1000, 2000, 5000, 10000],
                                   p=[0.30, 0.30, 0.20, 0.15, 0.05])
    impresiones = int(presupuesto / pr['cpc_mu'] * np.random.uniform(80, 120))
    ctr = max(0.1, np.random.normal(pr['ctr_mu'], pr['ctr_sd']))
    clics = int(impresiones * ctr / 100)
    tasa_conv = max(0.5, np.random.normal(pr['conv_mu'], 1.0))
    conversiones = int(clics * tasa_conv / 100)
    gasto = round(clics * pr['cpc_mu'] * np.random.uniform(0.85, 1.15), 2)
    ingresos = round(conversiones * np.random.uniform(40, 250), 2)
    roas = round(ingresos / gasto, 2) if gasto > 0 else 0

    registros.append({
        'fecha': fechas_inicio[i].date(),
        'mes': fechas_inicio[i].strftime('%Y-%m'),
        'plataforma': p,
        'tipo_campana': np.random.choice(tipos_campana),
        'segmento_edad': np.random.choice(segmentos),
        'presupuesto': presupuesto,
        'impresiones': impresiones,
        'clics': clics,
        'conversiones': conversiones,
        'gasto': gasto,
        'ingresos': ingresos,
        'ctr': round(ctr, 2),
        'cpc': round(gasto / clics, 3) if clics > 0 else 0,
        'roas': roas,
    })

df = pd.DataFrame(registros)
print(f'Dataset generado: {df.shape[0]} filas × {df.shape[1]} columnas')
df.head()

## 3. Resumen Estadístico General

In [ ]:
print('=== RESUMEN EJECUTIVO ===')
print(f"  Campañas analizadas : {len(df):,}")
print(f"  Gasto total         : ${df['gasto'].sum():,.0f}")
print(f"  Ingresos totales    : ${df['ingresos'].sum():,.0f}")
print(f"  ROAS promedio       : {df['roas'].mean():.2f}x")
print(f"  CTR promedio        : {df['ctr'].mean():.2f}%")
print(f"  Total conversiones  : {df['conversiones'].sum():,}")
print()
print('=== POR PLATAFORMA ===')
resumen = df.groupby('plataforma').agg(
    campanas=('gasto','count'),
    gasto_total=('gasto','sum'),
    ingresos_total=('ingresos','sum'),
    roas_prom=('roas','mean'),
    ctr_prom=('ctr','mean'),
    conversiones=('conversiones','sum')
).round(2)
print(resumen)

## 4. Visualizaciones

### 4.1 Gasto e Ingresos por Plataforma

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gasto total
gasto_plt = df.groupby('plataforma')['gasto'].sum().sort_values(ascending=False)
bars = axes[0].bar(gasto_plt.index, gasto_plt.values,
                   color=[PALETTE[p] for p in gasto_plt.index], edgecolor='white', linewidth=0.8)
axes[0].set_title('Gasto Total por Plataforma')
axes[0].set_ylabel('USD')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'${bar.get_height():,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# ROAS promedio
roas_plt = df.groupby('plataforma')['roas'].mean().sort_values(ascending=False)
bars2 = axes[1].bar(roas_plt.index, roas_plt.values,
                    color=[PALETTE[p] for p in roas_plt.index], edgecolor='white', linewidth=0.8)
axes[1].set_title('ROAS Promedio por Plataforma')
axes[1].set_ylabel('ROAS (x)')
axes[1].axhline(y=1, color='red', linestyle='--', linewidth=1, alpha=0.6, label='Break-even (1x)')
axes[1].legend()
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{bar.get_height():.2f}x', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Rendimiento Financiero por Plataforma', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 4.2 Evolución Mensual del CTR

In [ ]:
ctr_mensual = df.groupby(['mes', 'plataforma'])['ctr'].mean().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
for plat in plataformas:
    datos = ctr_mensual[ctr_mensual['plataforma'] == plat]
    ax.plot(datos['mes'], datos['ctr'], marker='o', linewidth=2,
            label=plat, color=PALETTE[plat], markersize=5)

ax.set_title('Evolución Mensual del CTR por Plataforma')
ax.set_ylabel('CTR (%)')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=45)
ax.legend(title='Plataforma', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

### 4.3 Distribución del ROAS por Tipo de Campaña

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
order = df.groupby('tipo_campana')['roas'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='tipo_campana', y='roas', order=order,
            palette='Set2', width=0.5, fliersize=3, ax=ax)
ax.axhline(y=1, color='red', linestyle='--', linewidth=1, alpha=0.5, label='Break-even')
ax.set_title('Distribución del ROAS por Tipo de Campaña')
ax.set_xlabel('Tipo de Campaña')
ax.set_ylabel('ROAS')
ax.legend()
plt.tight_layout()
plt.show()

### 4.4 Heatmap: CTR Promedio por Plataforma y Segmento de Edad

In [ ]:
pivot = df.pivot_table(values='ctr', index='plataforma', columns='segmento_edad', aggfunc='mean')

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, cbar_kws={'label': 'CTR (%)'}, ax=ax)
ax.set_title('CTR Promedio por Plataforma y Segmento de Edad')
ax.set_xlabel('Segmento de Edad')
ax.set_ylabel('Plataforma')
plt.tight_layout()
plt.show()

### 4.5 Relación Gasto vs Ingresos (scatter por plataforma)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for plat in plataformas:
    sub = df[df['plataforma'] == plat]
    ax.scatter(sub['gasto'], sub['ingresos'], alpha=0.35, s=25,
               color=PALETTE[plat], label=plat)

# Línea break-even ROAS = 1
max_val = max(df['gasto'].max(), df['ingresos'].max())
ax.plot([0, max_val], [0, max_val], 'r--', linewidth=1, alpha=0.6, label='Break-even (ROAS=1x)')
ax.set_title('Gasto vs Ingresos por Campaña')
ax.set_xlabel('Gasto (USD)')
ax.set_ylabel('Ingresos (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(title='Plataforma')
plt.tight_layout()
plt.show()

## 5. Conclusiones

Con base en el análisis visual de las 1,200 campañas:

1. **LinkedIn** lidera en ROAS (~6x) a pesar de tener el CPC más alto, lo que sugiere que su audiencia profesional convierte mejor.
2. **TikTok** destaca en CTR (>4.8%) y CPC más bajo, ideal para campañas de Awareness con presupuesto limitado.
3. **Instagram** es el canal con mayor volumen de gasto y conversiones absolutas; balance costo-beneficio sólido.
4. **Facebook** mantiene rendimiento estable; recomendable para Retargeting donde el CTR importa menos que la intención.
5. El segmento **25-34** muestra el mayor CTR en todas las plataformas → priorizar este segmento en campañas de Conversión.
6. Las campañas de **Retargeting** tienen la mediana de ROAS más alta, confirmando el valor de impactar a usuarios ya familiarizados con la marca.